# NurtureJoy Two-Stage Emotion Classifier — TF‑IDF + Logistic Regression

This notebook trains a **two-stage classifier**:

- **Stage 1:** POSITIVE / NEUTRAL / NEGATIVE  
- **Stage 2:** (only if NEGATIVE) STRESS / ANXIETY / LOW_MOOD / HIGH_DISTRESS

It then evaluates both stages and saves artifacts for deployment.

**Dataset expected columns:** `text`, `category`  
Default path: `./nurturejoy_emotion_with_preg_hardboundary.csv`

> Tip: Run on GPU if available for transformer notebooks.


In [ ]:

import os, re, numpy as np, pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

EMOTION_PATH = os.environ.get("EMOTION_PATH", "./nurturejoy_emotion_with_preg_hardboundary.csv")
assert os.path.exists(EMOTION_PATH), f"Missing dataset at: {EMOTION_PATH}"

def clean_text(t: str) -> str:
    t = str(t).strip()
    t = re.sub(r"\s+", " ", t)
    return t

LABELS6 = ["POSITIVE","NEUTRAL","ANXIETY","STRESS","LOW_MOOD","HIGH_DISTRESS"]
NEG4 = ["STRESS","ANXIETY","LOW_MOOD","HIGH_DISTRESS"]

df = pd.read_csv(EMOTION_PATH).dropna(subset=["text","category"]).copy()
df["text"] = df["text"].map(clean_text)
df = df[df["text"].str.len() >= 5]
df = df[df["category"].isin(LABELS6)].copy()

def to_stage1(lbl):
    if lbl == "POSITIVE": return "POSITIVE"
    if lbl == "NEUTRAL":  return "NEUTRAL"
    return "NEGATIVE"

df["stage1"] = df["category"].map(to_stage1)
df["stage2"] = df["category"].where(df["stage1"]=="NEGATIVE", other="NA")

print("Rows:", len(df))
print("Stage1 distribution:\n", df["stage1"].value_counts())
print("Stage2 (neg) distribution:\n", df[df.stage1=="NEGATIVE"]["category"].value_counts())
df.head(3)


In [ ]:
# Train Stage 1 and Stage 2 TF‑IDF models (fast baseline)


In [ ]:

from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

train_df, test_df = train_test_split(df, test_size=0.25, random_state=42, stratify=df["stage1"])

stage1 = Pipeline([
    ("tfidf", TfidfVectorizer(stop_words="english", ngram_range=(1,2), max_features=90000)),
    ("clf", LogisticRegression(max_iter=3000, class_weight="balanced"))
])
stage1.fit(train_df["text"], train_df["stage1"])

s1_pred = stage1.predict(test_df["text"])
print("=== STAGE 1 REPORT (POS/NEU/NEG) ===")
print(classification_report(test_df["stage1"], s1_pred))
print("Confusion matrix:\n", confusion_matrix(test_df["stage1"], s1_pred))


In [ ]:

train_neg = train_df[train_df["stage1"]=="NEGATIVE"].copy()
test_neg  = test_df[test_df["stage1"]=="NEGATIVE"].copy()

train_neg = train_neg[train_neg["category"].isin(NEG4)]
test_neg  = test_neg[test_neg["category"].isin(NEG4)]

stage2 = Pipeline([
    ("tfidf", TfidfVectorizer(stop_words="english", ngram_range=(1,2), max_features=90000)),
    ("clf", LogisticRegression(max_iter=3000, class_weight="balanced"))
])
stage2.fit(train_neg["text"], train_neg["category"])

s2_pred = stage2.predict(test_neg["text"])
print("=== STAGE 2 REPORT (NEG subtypes) ===")
print(classification_report(test_neg["category"], s2_pred, labels=NEG4))
print("Confusion matrix:\n", confusion_matrix(test_neg["category"], s2_pred, labels=NEG4))


In [ ]:

def predict_two_stage(text: str):
    text = clean_text(text)
    s1 = stage1.predict([text])[0]
    if s1 in ["POSITIVE","NEUTRAL"]:
        return s1
    return stage2.predict([text])[0]

tests = [
    "I’m excited today after my ultrasound!",
    "Today feels normal, just tired.",
    "I’m overwhelmed with work and appointments, it’s too much to handle.",
    "I can’t stop worrying about my glucose test results.",
    "I feel sad and lonely and I keep crying.",
    "I feel hopeless and unsafe with my thoughts."
]
for t in tests:
    print(predict_two_stage(t), " | ", t)


In [ ]:

import joblib, os
os.makedirs("artifacts_tfidf", exist_ok=True)
joblib.dump(stage1, "artifacts_tfidf/stage1_tfidf.joblib")
joblib.dump(stage2, "artifacts_tfidf/stage2_tfidf.joblib")
print("Saved: artifacts_tfidf/stage1_tfidf.joblib and artifacts_tfidf/stage2_tfidf.joblib")
